# Pronóstico de demanda — Planta KOF Bogotá (Fontibón)
### Recorrido guiado del pipeline (notebook)
Este notebook ejecuta y explica, paso a paso, el pipeline completo del proyecto.
**Los scripts en `scripts/` son la fuente única de verdad** — el notebook los
invoca con `%run`, de modo que no hay lógica duplicada.

Requisitos: entorno conda del repositorio (`conda env create -f environment.yml`,
`conda activate fontibon-pronostico`) y ejecutar el notebook **desde la raíz del repo**.

Ruta metodológica (ver `figuras/fig_metodologia.png` y el reporte, sec. 2–3):
datos reales → pruebas → Holt-Winters m=4 amortiguado → error/TS → distribución
→ Monte Carlo → multivariado → plan → ERP → QA.

## 0. Extracción reproducible de los datos
Lee los **17 Reportes de Resultados Trimestrales de Coca-Cola FEMSA** incluidos en
`referencias/descargas/` y extrae la fila *Colombia* de la tabla **Volumen**
(Refrescos | Agua | Garrafón | Otros | Total, en millones de cajas unidad).
Las columnas comparativas de los reportes 2022 aportan los trimestres de 2021
→ **21 trimestres reales (2021T1–2026T1)**.

In [ ]:
%run scripts/00_extraer_kof.py

## 1. Serie a escala de planta
`litros = MCU_categoría × 5,6781 L/CU × mezcla_empaque × participación_planta`.
La desagregación mensual usa pesos intra-trimestre documentados (el **total
trimestral es dato real**; solo la repartición interna es supuesta — supuesto S4).

In [ ]:
%run scripts/01_reconstruccion_datos.py

## 2. Pruebas estadísticas del flujo formal
- **Rachas**: ¿aleatoriedad? · **Kruskal-Wallis**: ¿medias iguales entre años? ·
**Levene**: ¿varianzas homogéneas? · **ACF de la serie diferenciada**: la
estacionalidad aparece como pico en el rezago 4 (Ljung-Box formaliza la significancia).
Teoría en el reporte, sec. 2.2.

In [ ]:
%run scripts/02_pruebas_estadisticas.py

## 3. Modelo Holt-Winters (multiplicativo, m=4, tendencia amortiguada)
Ecuaciones y justificación en el reporte, sec. 2.3. Incluye la **validación de
un paso**: el modelo entrenado hasta 2025T4 predice el 1T-2026 y se compara
contra el dato real observado. Horizonte final: **abr-2026 → mar-2027**.

In [ ]:
%run scripts/03_modelo_holt_winters.py

## 4. Error y señal de rastreo (backtest 5 trimestres reales)
MAD, CFE, MSE, MAPE y TS=CFE/MAD con límites ±4 (teoría: reporte sec. 2.4).
La TS toca −4 en el quiebre del impuesto 2025 — detección con causa asignable.

In [ ]:
%run scripts/04_evaluacion_errores.py

## 5–6. Plan en empaques/lotes y bases ERP (Odoo)
Jerarquías: 1.620 / 840 / 30 unidades por pallet. Lote mínimo = 1 pallet.
Los 9 CSVs de `erp_odoo/` (UdM, productos, componentes, LdM, empaques, MPS)
siguen la guía `erp_odoo/00_LEEME_ODOO.md`.

In [ ]:
%run scripts/05_inventario_lotes.py

In [ ]:
%run scripts/06_export_odoo.py

## 7. Libro Excel operativo (fórmulas)

In [ ]:
%run scripts/07_generar_excel.py

## 8–9. Distribución del error y simulación Monte Carlo
La Normal de los residuos relativos (n=21) se contrasta con **KS, Anderson-Darling
y χ²**; aceptada, se simulan **10.000 réplicas** (semilla 42) → bandas P5–P95.

In [ ]:
%run scripts/08_distribuciones.py

In [ ]:
%run scripts/09_montecarlo.py

## 10–11. Multivariado y diagrama de metodología
Correlación y PCA sobre las categorías reales: refrescos-agua co-mueven (r=0,87);
el **garrafón es casi independiente (r=0,12)** → programable en contraciclo.

In [ ]:
%run scripts/10_multivariado.py

In [ ]:
%run scripts/11_diagrama_flujo.py

## 12. Control de calidad automatizado (V1–V6)

In [ ]:
%run scripts/12_verificacion.py

## Resultados a la vista

In [ ]:
import pandas as pd, json
from IPython.display import Image, display
print("— Validación un-paso vs 1T-2026 real —")
print(json.dumps(json.load(open('data/validacion_2026T1.json')),indent=1))
print("\n— Pronóstico mensual (unidades) —")
display(pd.read_csv('data/pronostico_2026.csv').pivot(index='fecha',columns='producto',values='unidades'))
print("\n— Bandas Monte Carlo (litros/mes) —")
display(pd.read_csv('data/montecarlo.csv').head(6))
for f in ['fig_series','fig_montecarlo','fig_corr']:
    display(Image(filename=f'figuras/{f}.png'))